# Natural Language Processing



This project will give you practical experience using Natural Language Processing techniques. This project is in three parts:
- in part 1) you will use a dataset in a CSV file
- in part 2) you will use the Wikipedia API to directly access content
on Wikipedia.
- in part 3) you will make your notebook interactive


### Part 1)



- The CSV file is available at https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv
- The file contains a list of famous people and a brief overview.
- The goal of part 1) is to ...
  1. Pick one person from the list ( the reference person ) and output 10 other people who's overview are "closest" to the reference person in a Natural Language Processing sense
  1. Also output the sentiment of the overview of the reference person



### Part 2)



- For the same reference person that you chose in Part 1), use the Wikipedia API to access the whole content of the reference person's Wikipedia page.
- The goal of Part 2) is to ...
  1. Print out the text of the Wikipedia article for the reference person
  1. Determine the sentiment of the text of the Wikipedia page for the reference person
  1. Collect the text of the Wikipedia pages from the 10 nearest neighbors from Part 1)
  1. Determine the nearness ranking of these 10 people to your reference person based on their entire Wikipedia page
  1. Compare, i.e. plot,  the nearest ranking from Step 1) with the Wikipedia page nearness ranking.  A difference of the rank is one means of comparison.



### Part 3)


Make an interactive notebook where a user can choose or enter a name and the notebook displays the 10 closest individuals.

In addition to presenting the project slides, at the end of the presentation each student will demonstrate their code using a famous person suggested by the other students that exists in the DBpedia set.


## Data Collection

### Import

In [29]:
import duckdb
import pandas as pd
import os
import numpy as np
from google.colab import userdata


# TextBlob: cleans text (singularize) and scores sentiment
from textblob import TextBlob, Word

# TfidfVectorizer: turns each overview into a row of numbers (one column per word)
from sklearn.feature_extraction.text import TfidfVectorizer

# NearestNeighbors: finds the rows (people) closest to a reference row
from sklearn.neighbors import NearestNeighbors



## Convert CSV to Parquet

In [30]:
url = 'https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv'
url

'https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv'

In [31]:
# File name without the extension: 'NLP'
base_file = url.split("/")[-1].split(".")[0]
base_file

'NLP'

In [32]:
# Read the remote CSV and write it straight to a local parquet file (lesson 102's ETL step)
# No all_varchar here: DuckDB auto-detects types, so number columns stay numbers
con = duckdb.connect()
con.execute(f"""
    COPY (
        SELECT *
        FROM read_csv('{url}')
    )
    TO '{base_file}.parquet'
    (FORMAT PARQUET);
""")

! ls -l {base_file}.parquet

-rw-r--r-- 1 root root 49460246 Nov 25 14:06 NLP.parquet


### Push to Hugging Face

In [33]:
hf_account = os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account

'hmsealy'

In [34]:
hf_repo = os.environ["HF_REPO"] = "Data_Science-22"
hf_repo

'Data_Science-22'

In [35]:
hf_file = os.environ["HF_FILE"] = f"{base_file}.parquet"
hf_file

'NLP.parquet'

In [36]:
# Get the token from Colab Secrets. Only shows the first 3 and last 4 characters
_ = hf_token = os.environ["HF_TOKEN"] = userdata.get('hf_token')
f"{_[:3]} ... {_[-4:]}"

'hf_ ... kJTa'

In [37]:
! hf auth login --token ${HF_TOKEN}

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Colab` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [38]:
%%capture hf_upload
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

hf upload \
  --type dataset \
  "${TARGET_REPO}" \
  "${LOCAL_SOURCE}" \
  "${REMOTE_DESTINATION}"

In [39]:
print(hf_upload.stdout)

✓ Uploaded
  url: https://huggingface.co/datasets/hmsealy/Data_Science-22/commit/53b83f43a5d0a777ceb6ff484e323f69b898e28f



In [40]:
%%capture
# %%capture hides the long download log
!python -m textblob.download_corpora

In [41]:
# Pass 01 rule: use only the first 20 rows of the CSV
# We query the parquet file we already made, so this is fast
df = duckdb.query(f'select * from "{base_file}.parquet" limit 20').df()
df.shape

(20, 3)

In [42]:
ref_name = "Jonathan Hoefler"

# Find which row number Jonathan Hoefler is in
# .index gives the row labels where the condition is True; [0] takes the first one
ref_index = df.index[df["name"] == ref_name][0]
ref_index

np.int64(12)

In [43]:
# Turn each word into its singular form so "typefaces" and "typeface" count as the same word
def singularize_text(text):
    words = text.split()                               # split the overview into a list of words
    singles = [Word(w).singularize() for w in words]   # singularize each word
    return " ".join(singles)                           # glue the words back into one string

# Apply the function to every overview and keep the result in a new column
df["text_clean"] = df["text"].apply(singularize_text)

# Compare the original and cleaned text for the reference person
print(df.loc[ref_index, "text"][:200])
print(df.loc[ref_index, "text_clean"][:200])

jonathan hoefler born august 22 1970 is an american typeface designer hoefler pronounced heffler founded the hoefler type foundry in 1989 a type foundry in new york in 1999 hoefler began working with 
jonathan hoefler born august 22 1970 is an american typeface designer hoefler pronounced heffler founded the hoefler type foundry in 1989 a type foundry in new york in 1999 hoefler began working with 


In [44]:
# stop_words='english' drops filler words like "the", "is", "and"
tfidf_vec = TfidfVectorizer(stop_words="english")

# fit_transform learns the vocabulary and builds the matrix:
# 20 rows (people) x one column per unique word
tf_idf = tfidf_vec.fit_transform(df["text_clean"])
tf_idf.shape

(20, 1903)

In [45]:
# Teach NearestNeighbors where all 20 people sit in "word space"
nn = NearestNeighbors().fit(tf_idf)

# Ask for 11 neighbors, not 10: the closest match is always Jonathan himself (distance 0)
distances, indices = nn.kneighbors(
    X = tf_idf[ref_index],
    n_neighbors = 11,
)
distances, indices

(array([[0.        , 1.35020151, 1.3787278 , 1.38273374, 1.3837645 ,
         1.38534757, 1.3888504 , 1.38997952, 1.39005628, 1.39091275,
         1.39132888]]),
 array([[12,  5,  7, 19, 11,  3,  2,  1, 15, 14,  6]]))

In [46]:
# Pull out the people at those row numbers, and add their distances
neighbors = df.iloc[indices[0]][["name"]].copy()
neighbors["distance"] = distances[0]

# Drop the first row, which is Jonathan himself
neighbors = neighbors.iloc[1:]

# Add a rank column: 1 = closest
neighbors.insert(0, "rank", range(1, 11))
neighbors.reset_index(drop=True)

,rank,name,distance
0,1,Sam Henderson,1.350202
1,2,Trevor Ferguson,1.378728
2,3,John Angus Campbell,1.382734
3,4,Jenn Ashworth,1.383765
4,5,Franz Rottensteiner,1.385348
5,6,Harpdog Brown,1.388850
6,7,Alfred J. Lewy,1.389980
7,8,Joerg Steineck,1.390056
8,9,David Chernushenko,1.390913
9,10,Aaron LaCrate,1.391329


In [47]:
# Use the ORIGINAL text for sentiment, not the cleaned text
ref_blob = TextBlob(df.loc[ref_index, "text"])

# polarity: -1 (negative) to +1 (positive)
# subjectivity: 0 (factual) to 1 (opinion)
ref_blob.sentiment

Sentiment(polarity=0.19034090909090906, subjectivity=0.37683080808080804)